# Results & Retrospective
**FULL WRITE-UP · TRUE-HOLDOUT CONFIRMATION · LESSONS LEARNED**

---

## Content

- [1 · Objective](#1-objective)
- [2 · The modeling journey](#2-the-modeling-journey)
- [3 · Results — internal held-out test](#3-results-internal-held-out-test)
- [4 · Results — true AIM holdout](#4-results-true-aim-holdout)
- [5 · Robustness checks — leakage & overfitting](#5-robustness-checks-leakage-overfitting)
- [6 · Lessons learned](#6-lessons-learned)
- [7 · Recommendations](#7-recommendations)

**Kernaussage:** Auf dem echten Out-of-Sample-Set (den verdeckten AIM-Labels) erreicht der
getunte Champion **Bad-Buy-F1 0.409**. Das Modell generalisiert nahezu verlustfrei vom internen
Test auf den echten Holdout (Gap nur ~0.01) — ein systematischer Robustheits-Check (Leakage,
High-Cardinality-Overfitting) bestätigt, dass diese Zahl sauber ist.

Dieses Notebook ist das narrative Rückgrat des Projekts: Es konsolidiert die
Modellierungs-Reise, die finalen Zahlen (interner Test *und* echter Holdout), die
Robustheits-Checks und die Lessons Learned.

## 1 · Objective

Vorhersage, **vor dem Kauf**, ob ein bei einer Auktion gekauftes Gebrauchtfahrzeug ein "Bad Buy"
wird (ein Montagsauto, das nicht weiterverkauft werden kann). Die Zielvariable `IsBadBuy` ist
stark unbalanciert, Accuracy ist damit bedeutungslos, das Projekt optimiert stattdessen den
**F1-Score der Bad-Buy-Klasse**. Das Assessment setzte eine Hürde von **F1 > 0.40** auf einem
verdeckten Scoring-Set (`features_aim.csv`).

In [1]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, io, contextlib
from us_used_vehicle_resales.cleaning import clean_data
from us_used_vehicle_resales.features import engineer_features
from us_used_vehicle_resales.config_features_catalog import features_catalog
from us_used_vehicle_resales.config_models_catalog import models_catalog
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             roc_auc_score, confusion_matrix)

INT='../data/02_interim/'; RAW='../data/01_raw/'
pd.options.display.float_format='{:.4f}'.format

def _quiet(fn,*a,**k):
    with contextlib.redirect_stdout(io.StringIO()): return fn(*a,**k)

def load_prep(name):
    X=pd.read_parquet(f'{INT}features_{name}.parquet')
    y=pd.read_parquet(f'{INT}target_{name}.parquet').iloc[:,0]
    Xf=_quiet(engineer_features,_quiet(clean_data,X),print_status=False)
    return Xf, y.loc[Xf.index]

X_train,y_train=load_prep('train')
X_test, y_test =load_prep('test')

facts = pd.Series({
    'Training rows': len(X_train)+len(X_test),
    'Training columns (raw)': 33,
    'Bad-buy rate': f'{pd.concat([y_train,y_test]).mean():.2%}',
    'Held-out test rows': len(X_test),
    'Scoring set (AIM) rows': 7292,
    'Assessment bar': 'F1 > 0.40',
})
facts.to_frame('value')

,value
Training rows,65620
Training columns (raw),33
Bad-buy rate,12.35%
Held-out test rows,13124
Scoring set (AIM) rows,7292
Assessment bar,F1 > 0.40


## 2 · The modeling journey

1. **Baseline** — Logistische Regression auf 8 handverlesenen Features (Alter, Kilometerstand,
   Preisanker, zwei konstruierte Ratios, Auction, Make). Setzt den Boden.
2. **Systematisches Benchmark** — ein selbstgebauter **`ModelTracker`** fuhr **448 Experimente**
   über Feature-Sets × Modell-Familien (Logistische Regression Ridge/Lasso/Elastic-Net, Random
   Forest shallow/deep, HistGradientBoosting standard/aggressive), loggte
   F1/Recall/Precision/ROC-AUC pro Run, flaggte den besten und exportierte die gefittete
   Pipeline. Siehe [`05_experiment_framework.ipynb`](05_experiment_framework.ipynb) für die
   Catalogs und den Tracker selbst — die Infrastruktur, die einen schnellen, wiederholbaren
   Vergleich statt Ad-hoc-Trainingsläufen ermöglicht hat.
3. **Champion** — **Logistische Regression mit L1-Penalty** (`class_weight='balanced'`) auf dem
   vollen `all_in_with_noise`-Feature-Set. Zwei Befunde haben die Wahl getrieben:
   - **Kategoriales Signal ist essenziell:** Das Entfernen der kategorialen/konstruierten
     Features lässt F1 von ~0.38 auf ~0.29 einbrechen (siehe
     [`06_error_analysis.ipynb`](06_error_analysis.ipynb) dazu, wie stark sich das Modell auf
     `WheelType` stützt). Marktpreis-Numerik allein reicht nicht.
   - **Konsistent hoher Recall (~0.60)** bei moderater Precision — der richtige Trade-off für
     einen *Triage*-Use-Case, bei dem einen Bad Buy zu verpassen teurer ist als ein Fehlalarm.
4. **Threshold-Tuning** — bei balancierten Klassengewichten überflaggt der Default-Threshold von
   0.5. Der F1-optimale Operating Point auf dem Held-out-Test liegt bei **Threshold ≈ 0.65**.

**Eine Entscheidung, die sich bewährt hat:** Mehrere Random-Forest-Runs im Tracking-Log zeigten
F1 > 0.40, aber bei Recall ~0.3 und Precision ~0.7 — nutzlos für einen Triage-Filter, der Bad
Buys *fangen* muss. Diese wurden zurecht zugunsten der Logistischen Regression mit höherem
Recall verworfen.

## 3 · Results — internal held-out test

Alle Modelle evaluiert auf demselben **Held-out-Testset** (n = 13.124), Threshold 0.5 sofern
nicht anders angegeben. Das ist der gespeicherte Output von
[`04_evaluation.ipynb`](04_evaluation.ipynb), der Ergebnis-Single-Source-of-Truth — hier
geladen, nicht neu berechnet, damit die beiden Notebooks nie widersprüchlich sein können.

In [2]:
internal = pd.read_csv('../data/04_models/model_results_final_test.csv')
internal = internal.sort_values('F1', ascending=False).reset_index(drop=True)
internal

,Model,Features,Recall,Precision,F1,ROC_AUC
0,"LogReg Lasso (L1, balanced)",27,0.6039,0.2696,0.3728,0.7650
1,"Random Forest (deep, balanced)",27,0.6416,0.2425,0.3520,0.7491
2,Baseline LogReg (8 feat),8,0.6114,0.1874,0.2868,0.6662


In [3]:
feats=[f for f in features_catalog['all_in_with_noise'] if f in X_train.columns]
num=[f for f in feats if str(X_train[f].dtype).startswith(('int','float'))]
cat=[f for f in feats if f not in num]
pre=ColumnTransformer([('num',StandardScaler(),num),
                       ('cat',OneHotEncoder(handle_unknown='ignore'),cat)])
champ_pipe=Pipeline([('pre',pre),('clf',models_catalog['log_reg_lasso'])])
champ_pipe.fit(X_train[feats],y_train)
champ_proba_test=champ_pipe.predict_proba(X_test[feats])[:,1]

ths=np.linspace(0.2,0.85,66)
f1s=[f1_score(y_test,(champ_proba_test>=t).astype(int)) for t in ths]
best_t=float(ths[int(np.argmax(f1s))])
pred_t=(champ_proba_test>=best_t).astype(int)
print(f'Tuned threshold {best_t:.2f} -> F1 {f1_score(y_test,pred_t):.4f} '
      f'| Precision {precision_score(y_test,pred_t):.4f} | Recall {recall_score(y_test,pred_t):.4f}')

Tuned threshold 0.65 -> F1 0.4227 | Precision 0.4525 | Recall 0.3967


Der interne Test bestätigt: **LogReg Lasso ist das stärkste Modell**, und das Threshold-Tuning hebt es von F1 0.37 auf F1 0.42 — der grösste Einzelhebel in diesem Projekt.

## 4 · Results — true AIM holdout

Die verdeckte `target_aim.csv` wurde später wiedergewonnen, sodass die Modelle auf den **echten
Out-of-Sample-Labels** (7.292 Fahrzeuge, 863 Bad Buys) statt nur auf dem internen Test gescort
werden können. Sowohl Baseline als auch Champion werden hier neu trainiert (gleiche Catalogs,
gleicher `random_state=42`) und end-to-end gegen die rohe `features_aim.csv` gescort.

`target_aim.csv` selbst ist aus dem Repository ausgeschlossen und liegt nur lokal vor.

In [ ]:
base_feats=[f for f in features_catalog['baseline'] if f in X_train.columns]
base_num=[f for f in base_feats if str(X_train[f].dtype).startswith(('int','float'))]
base_cat=[f for f in base_feats if f not in base_num]
base_pre=ColumnTransformer([('num',StandardScaler(),base_num),
                            ('cat',OneHotEncoder(handle_unknown='ignore'),base_cat)])
base_pipe=Pipeline([('pre',base_pre),
                    ('clf',LogisticRegression(class_weight='balanced',random_state=42,max_iter=2000))])
base_pipe.fit(X_train[base_feats],y_train)

aim=pd.read_csv(f'{RAW}features_aim.csv',sep=';')
aim_f=_quiet(engineer_features,_quiet(clean_data,aim),print_status=False)
y_aim=pd.read_csv(f'{RAW}target_aim.csv').iloc[:,0]

def align(feat_list):
    out=aim_f.copy()
    for c in feat_list:
        if str(X_train[c].dtype).startswith(('int','float')):
            out[c]=pd.to_numeric(out[c],errors='coerce').fillna(0)
        else:
            out[c]=out[c].astype(str)
    return out[feat_list]

base_pred_aim=base_pipe.predict(align(base_feats))
base_f1_aim=f1_score(y_aim,base_pred_aim)
base_cm_aim=confusion_matrix(y_aim,base_pred_aim)
print(f'Baseline on AIM -> F1 {base_f1_aim:.4f}  confusion matrix {base_cm_aim.tolist()}')

In [5]:
champ_proba_aim=champ_pipe.predict_proba(align(feats))[:,1]

rows=[]
for t in [0.5, best_t]:
    pred=(champ_proba_aim>=t).astype(int)
    rows.append(dict(Threshold=t, F1=f1_score(y_aim,pred), Precision=precision_score(y_aim,pred),
                     Recall=recall_score(y_aim,pred), Flagged=pred.mean()))
champ_aim_results=pd.DataFrame(rows)
champ_aim_results

,Threshold,F1,Precision,Recall,Flagged
0,0.5000,0.3600,0.2595,0.5875,0.2680
1,0.6500,0.4089,0.4522,0.3731,0.0976


In [6]:
gap = internal.loc[internal.Model.str.contains('Lasso'), 'F1'].iloc[0] - champ_aim_results.loc[champ_aim_results.Threshold==0.5,'F1'].iloc[0]
print(f'Internal test F1 (0.5) vs AIM F1 (0.5) gap: {gap:.3f}')
print(f'Champion clears the 0.40 bar at the tuned threshold: '
      f'{champ_aim_results.loc[champ_aim_results.Threshold==best_t,"F1"].iloc[0]:.3f} > 0.40')

Internal test F1 (0.5) vs AIM F1 (0.5) gap: 0.013
Champion clears the 0.40 bar at the tuned threshold: 0.409 > 0.40


**Der interne Test-F1 und der AIM-F1 bei Threshold 0.5 unterscheiden sich um nur ~0.01** — das
Modell generalisiert fast perfekt. **Beim getunten Threshold knackt der Champion die
0.40-Hürde auf dem echten Holdout.** Diese 0.01-Lücke ist zugleich der stärkste empirische Beleg
gegen Leakage oder Overfitting: Ein aufgeblähter interner Score würde eine deutlich grössere
Lücke zum echten Holdout zeigen.

## 5 · Robustness checks — leakage & overfitting

Um sicherzugehen, dass die Ergebnisse sauber sind, wurde die Pipeline systematisch auf Data
Leakage und Overfitting geprüft:

- **Split vor dem Feature Engineering**, Transformer nur auf **Train** gefittet, **keine
  target-abgeleiteten Features**, **null doppelte Zeilen** über den Split hinweg → kein
  klassisches Leakage.
- **High-Cardinality-Memorization direkt getestet:** Das Entfernen von 2.082 memorisierbaren
  Kategorie-Levels (ZIP, Model, Sub-Model) verändert den Test-F1 um nur **0.003** → kein
  Overfitting.
- **Empirisch bestätigt** (§4): Die ~0.01-Lücke intern→Holdout auf echten
  Out-of-Sample-Daten ist der finale Beweis, dass interne Scores nicht aufgebläht sind.

**Es gibt kein Leakage.**

## 6 · Lessons learned

1. **Threshold-Tuning ist Teil des Deliverables, kein Nachgedanke.** Bei einer unbalancierten
   Zielvariable ist der Operating Point eine erstrangige Modellierungsentscheidung — der
   grösste Einzelhebel hier (0.36 → 0.41).
2. **Das *final gewählte* Modell einmal auf einem sauberen Held-out-Test evaluieren, bevor es
   rausgeht.** Eine Zahl, eine Source of Truth — siehe `04_evaluation.ipynb`.
3. **Ein hoher F1 bei unbrauchbarem Recall ist kein Gewinn.** Das vollständige
   Precision/Recall-Bild zu lesen statt nur die Headline-Metrik war der richtige Instinkt — das
   weiter so machen.
4. **Robustheits-Checks (Leakage, Overfitting) gehören systematisch zum Prozess**, nicht erst
   bei Verdacht — Vertrauen in eine Metrik beginnt mit dem Nachweis, dass sie sauber ist.
5. **Der selbstgebaute `ModelTracker` war ein echt guter Instinkt** — systematisch, geloggt,
   exportierbar. Für Produktion: MLflow / Weights & Biases machen das von der Stange, der
   Trade-off lässt sich also explizit benennen.

## 7 · Recommendations

- **Den getunten LogReg Lasso (Threshold ≈ 0.65) deployen** als **Triage-Filter**: Er flaggt
  ~10 % eines ungelabelten Batches bei ~0.45 Precision zur menschlichen Review — kein
  automatischer Reject.
- Einen fehlenden `WheelType` (`WheelType = Unknown`) als **erstrangiges Risiko-Flag beim
  Intake** behandeln — der stärkste Einzelprädiktor — aber mit einem zweiten Signal kombinieren
  für den in [`06_error_analysis.ipynb`](06_error_analysis.ipynb) identifizierten blinden
  Fleck: neuere, teurere Bad Buys ohne dieses Flag.
- Die zwei Batch-Level-Statistiken (Median-Imputation, `feat_price_cat`-Quantil-Bins) in die
  gefittete Pipeline einfalten, damit Single-Record-Scoring produktionssicher ist.

---

### Reproduzierbarkeit

| Artefakt | Ort |
|:---------|:---------|
| Ergebnis-SSoT (interne Test-Zahlen) | [`04_evaluation.ipynb`](04_evaluation.ipynb) |
| Error Analysis (Confusion Matrix, FN/FP-Segmente) | [`06_error_analysis.ipynb`](06_error_analysis.ipynb) |
| Engineering-Showcase (Catalogs, `ModelTracker`) | [`05_experiment_framework.ipynb`](05_experiment_framework.ipynb) |
| Robustheits-Checks (Leakage, Overfitting) | Abschnitt 5 dieses Notebooks |
| Modellvergleich / Threshold / Feature-Importance-Charts | [`public/img/`](../public/img/) |

<sub>Die verdeckte `target_aim.csv` bleibt ausserhalb des Repositorys; die AIM-Zahlen in §4
werden live in diesem Notebook dagegen berechnet.</sub>